<a href="https://colab.research.google.com/github/santoshkothapalli/genai/blob/main/Neo4j_MCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langgraph langchain langchain-openai neo4j mcp


In [100]:
from mcp.server.fastmcp import FastMCP
from neo4j import GraphDatabase

mcp = FastMCP("neo4j-move-server")
URI = "neo4j+s://demo.neo4jlabs.com"
USERNAME = "movies"
PASSWORD = "movies"

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))



In [95]:
def fetch_movies(tx, limit):
    query = """
    MATCH (m:Movie)
    RETURN m.title AS title, m.released AS released
    ORDER BY released DESC
    LIMIT $limit
    """
    result = tx.run(query, limit=limit)
    return [
        {
            "title": record["title"],
            "released": record["released"]
        }
        for record in result
    ]


@mcp.tool()
def get_movies(limit: int = 10):
    """
    Retrieve movies from Neo4j movie graph
    """
    query = """
    MATCH (m:Movie)
    RETURN m.title AS title, m.released AS released
    ORDER BY released DESC
    LIMIT $limit
    """
    with driver.session() as session:
        result = session.run(query, limit=limit)
        movies = [
            {
                "title": record["title"],
                "released": record["released"]
            }
            for record in result
        ]

    return {
        "movies": movies,
        "count": len(movies)
    }



In [96]:
from langchain.tools import tool

tools = [
    get_movies
]

In [97]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from google.colab import userdata
import os

api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key
llm = ChatOpenAI(model="gpt-4o")

agent = create_agent(
    llm,
    tools
)

In [101]:
response = agent.invoke(
    {"messages":[("user","Show movies available?")]}
)


In [102]:
import json

# Assuming the movie data is in the third message of the response (index 2) and it's a ToolMessage
# with the content as a JSON string.
if len(response['messages']) > 2 and 'content' in response['messages'][2].__dict__:
    tool_output_content = response['messages'][2].content
    try:
        movies_data = json.loads(tool_output_content)
        if 'movies' in movies_data:
            print("Movies available:")
            for movie in movies_data['movies']:
                print(f"- {movie['title']}")
        else:
            print("No 'movies' key found in the tool output.")
    except json.JSONDecodeError:
        print("Could not decode JSON from tool output.")
else:
    print("Could not find tool output in the response messages.")

Movies available:
- Cloud Atlas
- Ninja Assassin
- Frost/Nixon
- Speed Racer
- Charlie Wilson's War
- The Da Vinci Code
- RescueDawn
- V for Vendetta
- The Polar Express
- The Matrix Revolutions
